# Road Segmentation Training — Kaggle P100 / Google Colab

Trains **DeepLabV3-ResNet50** on the DeepGlobe satellite road dataset.

| Feature | Detail |
|---|---|
| Input size | 512 × 512 (resized from native 1024) |
| Batch size | 16 (P100 16 GB) |
| Loss | CrossEntropy + Dice, road class weighted ×10 |
| Training | Mixed precision (AMP), ReduceLROnPlateau on val IoU |
| Validation | 20 % held-out split, per-epoch IoU + F1 |
| Checkpoints | Saved every epoch; best model tracked separately |

**Resuming after disconnect:** re-run all cells — the checkpoint is reloaded automatically.

## Cell 1 — Platform
Change `PLATFORM` to `"colab"` when running on Google Colab (see instructions at the bottom of this notebook).

In [ ]:
import os

# ── Change this to "colab" when running on Google Colab ──────────────────────
PLATFORM = "kaggle"

if PLATFORM == "kaggle":
    OUTPUT_DIR      = "/kaggle/working"
    CHECKPOINT_PATH = "/kaggle/working/checkpoint.pth"
    FINAL_MODEL     = "/kaggle/working/DeeplabsV3_road_final.pth"
    BEST_MODEL      = "/kaggle/working/best_model.pth"
else:
    OUTPUT_DIR      = "/content"
    CHECKPOINT_PATH = "/content/checkpoint.pth"
    FINAL_MODEL     = "/content/DeeplabsV3_road_final.pth"
    BEST_MODEL      = "/content/best_model.pth"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Platform : {PLATFORM}")
print(f"Output   : {OUTPUT_DIR}")

## Cell 2 — Install Dependencies

In [ ]:
%pip install kagglehub torch torchvision tqdm opencv-python scikit-image -q
print("✅ Dependencies installed")

## Cell 3 — GPU Check

> **Kaggle:** Settings → Accelerator → **GPU P100**  
> **Colab:** Runtime → Change runtime type → GPU (T4 or better)

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU     : {name}  ({vram:.1f} GB VRAM)")
    if "P100" in name:
        print("✅ Kaggle P100 detected — batch_size=16 at 512px is safe")
    elif "T4" in name:
        print("✅ Colab T4 detected — batch_size=8 at 512px recommended")
    else:
        print(f"⚡ GPU ready")
else:
    raise RuntimeError("❌ No GPU found. Enable GPU in accelerator settings before continuing.")

## Cell 4 — Dataset Download

**Kaggle:** The dataset is downloaded automatically via `kagglehub`.  
**Colab:** You need a Kaggle API key first — see instructions at the bottom of this notebook.

In [ ]:
import kagglehub

path = kagglehub.dataset_download("balraj98/deepglobe-road-extraction-dataset")
print(f"Dataset path: {path}")

# DeepGlobe layout: images and masks both in /train/ with _sat.jpg / _mask.png naming
train_dir = os.path.join(path, "train")
if not os.path.isdir(train_dir):
    train_dir = path

sat_files = [f for f in os.listdir(train_dir) if f.endswith("_sat.jpg")]
print(f"Satellite images found: {len(sat_files)}")
assert len(sat_files) > 0, "No *_sat.jpg images found — check dataset path"

## Cell 5 — Hyperparameters

Tune here before starting training.

In [ ]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE    = 512   # resize from native 1024×1024 → 4× less memory
BATCH_SIZE  = 16    # P100 16 GB: safe at 512px.  T4: use 8.  A100: try 32.
NUM_EPOCHS  = 10
LR          = 1e-4
VAL_SPLIT   = 0.2   # 20% held-out validation
SEED        = 42
NUM_WORKERS = 4
USE_CLAHE   = True  # CLAHE contrast enhancement — improves road visibility
BACKBONE    = "resnet50"  # "resnet50" or "resnet101" (slower, slightly more accurate)
USE_AMP     = torch.cuda.is_available()  # mixed precision — ~50% faster on GPU
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"IMG_SIZE={IMG_SIZE}  BATCH={BATCH_SIZE}  EPOCHS={NUM_EPOCHS}")
print(f"AMP={USE_AMP}  CLAHE={USE_CLAHE}  BACKBONE={BACKBONE}")
print(f"Device: {DEVICE}")

# Estimated time (P100, batch=16, 512px, ~6200 train images):
batches_per_epoch = int(len(sat_files) * (1 - VAL_SPLIT)) // BATCH_SIZE
est_min = batches_per_epoch * 2 // 60  # ~2s per batch with AMP
print(f"\nEstimated ~{est_min}-{est_min*2} min/epoch → {est_min*NUM_EPOCHS//60}–{est_min*2*NUM_EPOCHS//60} hrs total")

## Cell 6 — Dataset Class + Train/Val Split

In [ ]:
import cv2
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF


def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):
    img_np = np.array(image)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    lab_enhanced = cv2.merge((clahe.apply(l), a, b))
    return Image.fromarray(cv2.cvtColor(lab_enhanced, cv2.COLOR_LAB2RGB))


class RoadSegmentationDataset(Dataset):
    def __init__(self, data_dir, file_list, img_size=512, augment=True, use_clahe=True):
        self.data_dir  = data_dir
        self.files     = sorted(file_list)
        self.augment   = augment
        self.use_clahe = use_clahe
        self.resize    = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225],
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        name  = self.files[idx]
        image = Image.open(os.path.join(self.data_dir, name)).convert("RGB")
        mask  = Image.open(os.path.join(self.data_dir, name.replace("_sat.jpg", "_mask.png"))).convert("L")

        # Joint resize (keeps image/mask aligned)
        image = self.resize(image)
        mask  = self.resize(mask)

        if self.use_clahe:
            image = apply_clahe(image)

        # Joint spatial augmentation
        if self.augment:
            if random.random() > 0.5:
                image, mask = TF.hflip(image), TF.hflip(mask)
            if random.random() > 0.5:
                image, mask = TF.vflip(image), TF.vflip(mask)
            if random.random() > 0.5:
                angle = random.choice([90, 180, 270])
                image, mask = TF.rotate(image, angle), TF.rotate(mask, angle)

        image = self.normalize(self.to_tensor(image))
        mask  = (self.to_tensor(mask) > 0.5).long().squeeze(0)
        return image, mask


# ── Deterministic 80/20 train/val split ───────────────────────────────────────
all_files = sorted(f for f in os.listdir(train_dir) if f.endswith("_sat.jpg"))
rng = random.Random(SEED)
shuffled = all_files[:]
rng.shuffle(shuffled)
split      = int(len(shuffled) * (1 - VAL_SPLIT))
train_files = shuffled[:split]
val_files   = shuffled[split:]

train_ds = RoadSegmentationDataset(train_dir, train_files, IMG_SIZE, augment=True,  use_clahe=USE_CLAHE)
val_ds   = RoadSegmentationDataset(train_dir, val_files,   IMG_SIZE, augment=False, use_clahe=USE_CLAHE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_ds)} images  ({len(train_loader)} batches)")
print(f"Val  : {len(val_ds)} images  ({len(val_loader)} batches)")

## Cell 7 — Model

In [ ]:
import torch.nn as nn
from torchvision.models.segmentation import deeplabv3_resnet50, deeplabv3_resnet101


def build_model(backbone="resnet50"):
    """DeepLabV3 with ImageNet-pretrained backbone, 2-class road head."""
    if backbone == "resnet101":
        model = deeplabv3_resnet101(weights="DEFAULT")
    else:
        model = deeplabv3_resnet50(weights="DEFAULT")
    model.classifier[4]     = nn.Conv2d(256, 2, kernel_size=1)
    model.aux_classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
    return model


model = build_model(BACKBONE).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"✅ DeepLabV3-{BACKBONE} loaded on {DEVICE}  ({total_params:.1f} M params)")

## Cell 8 — Training Loop

Runs all epochs. Re-run this cell after a disconnect to resume from the last checkpoint.

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm


# ── Loss ──────────────────────────────────────────────────────────────────────
class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth

    def forward(self, predict, target):
        predict = torch.softmax(predict, dim=1)[:, 1, :, :]
        target  = target.float()
        inter   = (predict * target).sum(dim=(1, 2))
        union   = predict.sum(dim=(1, 2)) + target.sum(dim=(1, 2))
        return 1 - ((2. * inter + self.smooth) / (union + self.smooth)).mean()


class_weights = torch.tensor([1.0, 10.0]).to(DEVICE)  # road class weighted 10×
ce_loss       = nn.CrossEntropyLoss(weight=class_weights)
dice_loss     = DiceLoss()


def criterion(out, tgt):
    return ce_loss(out, tgt) + dice_loss(out, tgt)


optimizer = Adam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)


# ── Checkpoint helpers ─────────────────────────────────────────────────────────
def save_checkpoint(epoch, model, optimizer, scaler, val_iou, path):
    torch.save({
        "epoch":                epoch,
        "model_state_dict":     model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict":    scaler.state_dict(),
        "val_iou":              val_iou,
        "backbone":             BACKBONE,
        "img_size":             IMG_SIZE,
    }, path)


def load_checkpoint(path, model, optimizer, scaler):
    if not os.path.exists(path):
        return 0, 0.0
    ckpt = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start    = ckpt["epoch"] + 1
    best_iou = ckpt.get("val_iou", 0.0)
    print(f"✅ Resumed from epoch {start}  (best val IoU so far: {best_iou:.4f})")
    return start, best_iou


# ── Validation ────────────────────────────────────────────────────────────────
def validate_epoch(model, loader):
    model.eval()
    ious, f1s = [], []
    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = model(imgs)["out"]
            preds = torch.argmax(out, dim=1).cpu()
            for i in range(preds.size(0)):
                p = preds[i].view(-1)
                t = masks[i].view(-1)
                tp = ((p == 1) & (t == 1)).sum().item()
                fp = ((p == 1) & (t == 0)).sum().item()
                fn = ((p == 0) & (t == 1)).sum().item()
                ious.append(tp / (tp + fp + fn + 1e-7))
                prec = tp / (tp + fp + 1e-7)
                rec  = tp / (tp + fn + 1e-7)
                f1s.append(2 * prec * rec / (prec + rec + 1e-7))
    model.train()
    return float(sum(ious) / len(ious)), float(sum(f1s) / len(f1s))


# ── Run ────────────────────────────────────────────────────────────────────────
start_epoch, best_iou = load_checkpoint(CHECKPOINT_PATH, model, optimizer, scaler)

print(f"\n🚀 Training on {DEVICE}  |  AMP={USE_AMP}  |  batch={BATCH_SIZE}  |  img={IMG_SIZE}px")
print(f"   Epochs {start_epoch + 1} → {NUM_EPOCHS}")
print("=" * 65)

model.train()
for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_loss = 0.0
    bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", unit="batch")

    for i, (imgs, masks) in enumerate(bar):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out  = model(imgs)["out"]
            loss = criterion(out, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        bar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{epoch_loss/(i+1):.4f}")

    avg_loss          = epoch_loss / len(train_loader)
    val_iou, val_f1   = validate_epoch(model, val_loader)
    scheduler.step(val_iou)
    lr_now = optimizer.param_groups[0]["lr"]

    print(f"Epoch {epoch+1:>2}/{NUM_EPOCHS}  "
          f"train_loss={avg_loss:.4f}  val_iou={val_iou:.4f}  val_f1={val_f1:.4f}  lr={lr_now:.2e}")

    save_checkpoint(epoch, model, optimizer, scaler, val_iou, CHECKPOINT_PATH)

    if val_iou > best_iou:
        best_iou = val_iou
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch":    epoch,
            "val_iou":  val_iou,
            "backbone": BACKBONE,
            "img_size": IMG_SIZE,
        }, BEST_MODEL)
        print(f"   ⭐ New best IoU: {best_iou:.4f} — saved best_model.pth")

print(f"\n✅ Training complete!  Best val IoU: {best_iou:.4f}")

In [ ]:
import shutil

# Save the final model (last epoch)
torch.save({
    "model_state_dict": model.state_dict(),
    "epoch":    NUM_EPOCHS,
    "val_iou":  val_iou,
    "backbone": BACKBONE,
    "img_size": IMG_SIZE,
}, FINAL_MODEL)
print(f"💾 Final model  → {FINAL_MODEL}")
print(f"💾 Best model   → {BEST_MODEL}")

# Zip everything for convenience
zip_path = os.path.join(OUTPUT_DIR, "road_seg_models")
shutil.make_archive(zip_path, "zip", OUTPUT_DIR,
                    base_dir=None)
# Only zip the .pth files, not the whole directory
import zipfile
zip_file = zip_path + ".zip"
with zipfile.ZipFile(zip_file, "w") as zf:
    for f in [FINAL_MODEL, BEST_MODEL, CHECKPOINT_PATH]:
        if os.path.exists(f):
            zf.write(f, os.path.basename(f))
print(f"📦 Zipped models → {zip_file}")

# ── Platform-specific download ─────────────────────────────────────────────────
if PLATFORM == "colab":
    from google.colab import files
    print("⬇️  Downloading files to your computer...")
    files.download(FINAL_MODEL)
    files.download(BEST_MODEL)
    files.download(zip_file)
else:
    print("\n📂 Kaggle: open the Output panel (right sidebar) to download your files.")
    print(f"   Files saved in: {OUTPUT_DIR}")
    for f in [FINAL_MODEL, BEST_MODEL, zip_file]:
        if os.path.exists(f):
            size = os.path.getsize(f) / 1e6
            print(f"   ✅ {os.path.basename(f)}  ({size:.1f} MB)")

---
## Using this notebook on Google Colab

### Step 1 — Download the notebook from Kaggle
Go to your Kaggle notebook → **File → Download notebook** → saves as `colab_training.ipynb`.

### Step 2 — Upload to Colab
Go to [colab.research.google.com](https://colab.research.google.com) → **File → Upload notebook** → select the `.ipynb` file.

### Step 3 — Enable GPU
**Runtime → Change runtime type → Hardware accelerator → GPU (T4 or better)** → Save.

### Step 4 — Change one variable
In **Cell 1** change:
```python
PLATFORM = "kaggle"
```
to:
```python
PLATFORM = "colab"
```
This switches output paths and enables `files.download()` at the end.

### Step 5 — Set up Kaggle API credentials (for dataset download)
The dataset downloads via `kagglehub`, which needs your Kaggle API key.  
Run this in a new cell **before** Cell 4:
```python
import os
os.environ["KAGGLE_USERNAME"] = "your_kaggle_username"
os.environ["KAGGLE_KEY"]      = "your_api_key"
```
Get your key from: **kaggle.com → Account → Settings → API → Create New Token** → downloads `kaggle.json`.

### Step 6 — Run all cells
**Runtime → Run all**. Training starts automatically.  
At the end, your browser will download `DeeplabsV3_road_final.pth` and `best_model.pth` automatically.

### Batch size on Colab T4 (15 GB VRAM)
Change `BATCH_SIZE = 16` → `BATCH_SIZE = 8` in Cell 5 for T4.  
Estimated time with T4 + batch=8: ~15–20 min/epoch → ~2.5–3 hrs for 10 epochs.

---
### After training — using the model locally

Put the downloaded `.pth` file at `models/checkpoints/DeeplabsV3_road_final.pth` in your project folder, then run:
```bash
python run_inference.py
```
Or launch the app:
```bash
python app.py
```

## Cell 9 — Save & Download Model

**Kaggle:** Files are automatically saved to `/kaggle/working/` and appear in the **Output** panel on the right. Download from there.  
**Colab:** This cell triggers a browser download for both model files.